Imports

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.stem import LancasterStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.sparse import csr_matrix

Parameters

In [ ]:
PCA_number_components=2500 #still need to tuned
numberStopwords=50
stemmization_Flag=True
lemmization_Flag=True

Load data

In [ ]:
train_data = pd.read_csv("data_supervised.csv", encoding="utf8")
target = pd.read_csv("target_supervised.csv", encoding="utf8")
unsupervised_data=pd.read_csv("data_unsupervised.csv", encoding="utf8")
#for each dataset we print for each column the number of elements missing
print(train_data.isnull().sum())
print(target.isnull().sum())
print(unsupervised_data.isnull().sum())


Only unsupervised_data.csv has 2 comments missing
deletion of the missing comments

In [ ]:
unsupervised_data=unsupervised_data.dropna()
print(unsupervised_data.isnull().sum())

Count duplicates

In [ ]:
print(train_data.duplicated().sum())
print(unsupervised_data.duplicated().sum())  
print(target.duplicated().sum())  

Deletion of duplicated rows

In [ ]:
train_data=train_data.drop_duplicates()
unsupervised_data=unsupervised_data.drop_duplicates()
print(train_data.duplicated().sum())
print(unsupervised_data.duplicated().sum())  
print(target.duplicated().sum())  

Empirical distribution of the number of characterst in each comment

In [ ]:
#add  a length column to every row
train_data['length'] = train_data['body'].astype(str).apply(len)

#we can define a series that has the length as an index and the number of comments that match the length as the value
countForLenght=train_data.groupby('length').count()['body']

print(countForLenght.max)

x = countForLenght.index
y =  countForLenght.values

df = pd.DataFrame({
    'length' : x, #'length' : x[0:1000], 
    'number of comments' : y#[0:1000]
})

plt.figure()
sns.lineplot(
    data = df, # which dataframe you want to use as the source data
    x = 'length', # which column to use for x-axis
    y = 'number of comments', # which column to use for y-axis
    ax = plt.gca() # assign this plot to current axis, not mandatory in this case
)
plt.show()


In [ ]:
plt.figure()
sns.lineplot(
    train_data.groupby('author').count()['subreddit']
)
plt.show()


In [ ]:
#train_data.groupby('author').count()['subreddit'].sort_values()

In [ ]:
#printing the number of males and females
print(target.groupby('gender').count())

#printing the number of comments for males and females
print(pd.merge(target,train_data).groupby('gender').count())

In [ ]:
train_data=train_data.applymap(lambda x: x.lower() if isinstance(x, str) else x)
target=target.applymap(lambda x: x.lower() if isinstance(x, str) else x)
unsupervised_data.applymap(lambda x: x.lower() if isinstance(x, str) else x)


In [ ]:
#train_data['body'].str.split().explode()

MostFrequentWords = train_data['body'].str.split().explode().value_counts()[0:numberStopwords].index.tolist()
print(MostFrequentWords)
for word in MostFrequentWords:
    train_data['body']=train_data['body'].str.replace(rf'\b{word}\b', '', regex=True)
MostFrequentWords = train_data['body'].str.split().explode().value_counts()[0:numberStopwords].index.tolist()
print(MostFrequentWords)


In [ ]:
#lemmization

# Function to tokenize and lemmatize a single text
def lemmatize_text(text,lemmatizer):
    # Tokenize the text into words
    tokens = word_tokenize(text)
    
    # Lemmatize each token
    lemmatized_words = []
    for word in tokens:
        lemmatized_word = lemmatizer.lemmatize(word)
        lemmatized_words.append(lemmatized_word)
    
    # Join the lemmatized words back into a single string
    lemmatized_text = ' '.join(lemmatized_words)
    return lemmatized_text

def lemmization(df):
    # Initialize the lemmatizer
    lemmatizer = WordNetLemmatizer()

    # Apply the lemmatization function to the DataFrame column
    for index, row in train_data.iterrows():
        lemmatized_result = lemmatize_text(row['body'],lemmatizer)  # Process each row
        df.at[index, 'body'] = lemmatized_result

if lemmization_Flag:
    lemmization(train_data)

In [ ]:
# Print the resulting DataFrame
print(train_data['body'][0])

In [ ]:
#stemming
def stemmer_text(text,stemmer):
    # Tokenize the text into words
    tokens = word_tokenize(text)
    #tokens = text.split(' ')#what if we don't tokenize?
    stemmer_words = []
    for word in tokens: 
        stemmer_words.append(stemmer.stem(word))
    
    stemmed_text = ' '.join(stemmer_words)
    return stemmed_text

def stemmization(df):
    stemmer = LancasterStemmer()

    for index, row in train_data.iterrows():
        stemmed_result = stemmer_text(row['body'],stemmer)  # Process each row
        df.at[index, 'body'] = stemmed_result

if stemmization_Flag:
    stemmization(train_data)

In [ ]:
print(train_data['body'][0])

In [ ]:
#grouping by the author
#TODO controllare se sta roba serve
train_data=train_data[['author','body']].groupby('author').agg(' '.join).reset_index()

vectorizerCount = CountVectorizer()
vectorizerCount.fit(train_data['body'])
print(sorted(vectorizerCount.vocabulary_))
vectorizerCount = vectorizerCount.transform(train_data['body'])

In [ ]:
print(vectorizerCount.shape)
#print(csr_matrix(vectorizerCount.toarray()))
#print(len(vectorizerCount.vocabulary_))


In [ ]:
vectorizerTfidf = TfidfVectorizer()
vectorizerTfidf.fit(train_data['body'])
#print(sorted(vectorizerTfidf.vocabulary_))
vectorizerTfidf = vectorizerTfidf.transform(train_data['body'])

#vectorizerTfidf.fit(unsupervised_data['body'])
#print(sorted(vectorizerTfidf.vocabulary_))
#vectorizerTfidf = vectorizerTfidf.transform(unsupervised_data['body']))

In [ ]:
#print(csr_matrix(vectorizerTfidf.toarray()))
#boh = vectorizerTfidf.toarray()

4. Reduce the complexity of your dataset.

In [ ]:
pca=PCA(n_components=PCA_number_components,random_state=42,)
pca.fit(vectorizerTfidf)


In [ ]:
# describe how much of the dataset variability is indicated by a given amount of features
explained_variance = pca.explained_variance_ratio_

# evaluate the total dataset variability while increasing the variables
cumul_exp_var = np.cumsum(explained_variance)

# percentage value to better understand the best number of components
perc_cumul_exp_var = cumul_exp_var * 100

# make the plot of cumulative explained variance wrt number of components

plt.figure(figsize=(5, 3.5))
plt.plot(perc_cumul_exp_var, marker='o')
plt.xlabel('# Principal Components (PCs)')
plt.ylabel('Cumulative explained variance [%]')
plt.xticks([i for i in range(PCA_number_components)], [i for i in range(1,PCA_number_components+1)])
plt.grid()
plt.title(f'{(PCA_number_components/2)-1} PCs explain {round(perc_cumul_exp_var[int(PCA_number_components/2)], 2)}% of $\sigma^2$')
plt.tight_layout()
plt.show()

print(perc_cumul_exp_var[-1])
pca_result = pca.transform(vectorizerTfidf)

dfPCA=pd.DataFrame(pca_result)
dfPCA.to_csv("PCA.csv")